In [40]:
import pandas as pd
import numpy as np

In [41]:
ufos = pd.read_csv('./data/ufos.csv')
ufos.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [42]:
ufos = pd.DataFrame({'Seconds': ufos['duration (seconds)'], 'Country': ufos['country'],'Latitude': ufos['latitude'],'Longitude': ufos['longitude']})

ufos.Country.unique()

<StringArray>
['us', nan, 'gb', 'ca', 'au', 'de']
Length: 6, dtype: str

In [43]:
ufos.dropna(inplace=True)
ufos = ufos[(ufos['Seconds'] >= 1) & (ufos['Seconds'] <= 60)]
ufos.info()

<class 'pandas.DataFrame'>
Index: 25863 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    25863 non-null  float64
 1   Country    25863 non-null  str    
 2   Latitude   25863 non-null  float64
 3   Longitude  25863 non-null  float64
dtypes: float64(3), str(1)
memory usage: 1010.3 KB


In [44]:
from sklearn.preprocessing import LabelEncoder

In [45]:
ufos['Country'] = LabelEncoder().fit_transform(ufos['Country'])

In [46]:
ufos.head()

,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
23,60.0,4,45.582778,-122.352222
24,3.0,3,51.783333,-0.783333


In [47]:
#building the model
from sklearn.model_selection import train_test_split

In [48]:
selected_features = ['Seconds','Latitude','Longitude']

X = ufos[selected_features]
y = ufos['Country']

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size= 0.2, random_state= 0)

In [49]:
#training the model
from sklearn.metrics import accuracy_score, classification_report

#sclaing features to improve model performnce, without doing that the logistic regression optimization algorithm with not converge at the default 100 iterations
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
model.fit(X_train,y_train)
predictions = model.predict(X_test)

In [50]:
#Evaluating the model
print ( classification_report(y_test, predictions))
print('Preidcted labels: ', predictions)
print ('Accuracy:', accuracy_score(y_test, predictions))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.84      0.46      0.59       250
           2       1.00      0.12      0.22         8
           3       0.94      1.00      0.97       131
           4       0.97      1.00      0.98      4743

    accuracy                           0.97      5173
   macro avg       0.95      0.72      0.75      5173
weighted avg       0.97      0.97      0.96      5173

Preidcted labels:  [4 4 4 ... 3 4 4]
Accuracy: 0.9682969263483472


In [51]:
new_data = pd.DataFrame([[50, 44, -12]], columns=['Seconds', 'Latitude', 'Longitude'])
predictions = model.predict(new_data)
print(predictions)

[2]


In [52]:
#pickling the model
import pickle
model_filename = 'ufo-model.pkl'
pickle.dump(model, open(model_filename, 'wb'))

#testing the pickled model
model = pickle.load(open('ufo-model.pkl', 'rb'))
new_data = pd.DataFrame([[50, 44, -12]], columns=['Seconds', 'Latitude', 'Longitude'])
predictions = model.predict(new_data)
print(predictions)

[2]
